# AudioRestore Demo: Text-Controlled Audio Restoration

**CS 614 Final Project — Gary Pham (gp492) — Drexel University**

Fine-tuned [SonicMaster](https://github.com/AMAAI-Lab/SonicMaster) (0.9B params) for codec audio restoration controlled by natural language prompts.

## Setup

1. **Runtime**: Open in Google Colab with an **A100 GPU**, then *Run All*
2. **Data & Checkpoints**: The next cell will mount your Google Drive and copy data from the shared folder.
   - **First time only**: Open the [shared Drive folder](https://drive.google.com/drive/folders/1eyxwTykveOsbY3XybfIVmWsd1kFF4AAJ?usp=sharing), right-click → **"Organize" → "Add shortcut"** → save to **My Drive**. This makes it visible after mounting.
3. **HuggingFace Token**: Add `HF_TOKEN` as a Colab secret (sidebar key icon) to download the Stable Audio VAE

## Demo outline
1. Load the fine-tuned model and VAE
2. Restore audio degraded at 32 / 64 / 128 kbps MP3
3. Compare spectrograms and compute SDR / SI-SNR
4. Test different text prompts on the same degraded audio

In [ ]:
import os, sys

try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    PROJECT_ROOT = '/content/CS614_Project'
    SM_DIR = '/content/sonicmaster'
    !pip install diffusers>=0.30.0 transformers>=4.44.0 accelerate>=0.34.2 -q
    !pip install safetensors datasets librosa soundfile tqdm pandas pyyaml -q
    !pip install huggingface_hub -q

    # --- Mount Google Drive & use data directly (no copy needed) ---
    from google.colab import drive
    drive.mount('/content/drive')

    from glob import glob as gglob
    DRIVE_SHARED = None
    ckpt_patterns = [
        '/content/drive/MyDrive/*/outputs/finetune_codec/best/model.safetensors',
        '/content/drive/MyDrive/*/checkpoints/model.safetensors',
        '/content/drive/MyDrive/*/*/outputs/finetune_codec/best/model.safetensors',
        '/content/drive/MyDrive/*/*/checkpoints/model.safetensors',
    ]
    for pattern in ckpt_patterns:
        for match in gglob(pattern):
            parts = match.replace('/content/drive/MyDrive/', '').split('/')
            depth = parts.index('outputs') if 'outputs' in parts else parts.index('checkpoints')
            DRIVE_SHARED = '/content/drive/MyDrive/' + '/'.join(parts[:depth])
            break
        if DRIVE_SHARED:
            break

    if DRIVE_SHARED:
        PROJECT_ROOT = DRIVE_SHARED
        print(f'Using data directly from Drive: {PROJECT_ROOT}')
    else:
        os.makedirs(PROJECT_ROOT, exist_ok=True)
        print('Could not auto-detect project data. Listing MyDrive root:')
        !ls /content/drive/MyDrive/
        print(
            '\nERROR: Could not find project data in Google Drive.\n'
            '  1. Open: https://drive.google.com/drive/folders/1eyxwTykveOsbY3XybfIVmWsd1kFF4AAJ\n'
            '  2. Right-click the folder -> Organize -> Add shortcut -> My Drive\n'
            '  3. Re-run this cell.'
        )
else:
    cwd = os.getcwd()
    if os.path.basename(cwd) == 'Project':
        PROJECT_ROOT = cwd
    elif os.path.isdir(os.path.join(cwd, 'Project')):
        PROJECT_ROOT = os.path.join(cwd, 'Project')
    else:
        PROJECT_ROOT = cwd
    SM_DIR = os.path.join(PROJECT_ROOT, 'sonicmaster')

if not os.path.isdir(SM_DIR):
    !git clone https://github.com/AMAAI-Lab/SonicMaster.git {SM_DIR}
sys.path.insert(0, SM_DIR)

import torch, torchaudio, numpy as np, librosa, librosa.display
import soundfile as sf, matplotlib.pyplot as plt, yaml, glob
from IPython.display import Audio, display

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Environment: {"Colab" if IS_COLAB else "Local"} | Device: {device}')

In [ ]:
from model import TangoFlux
from safetensors.torch import load_file
from diffusers import AutoencoderOobleck

CONFIG_PATH = os.path.join(SM_DIR, 'configs', 'tangoflux_config.yaml')
for ckpt_candidate in [
    os.path.join(PROJECT_ROOT, 'outputs', 'finetune_codec', 'best', 'model.safetensors'),
    os.path.join(PROJECT_ROOT, 'checkpoints', 'model.safetensors'),
]:
    if os.path.exists(ckpt_candidate):
        CKPT_PATH = ckpt_candidate
        break
else:
    raise FileNotFoundError(
        'Checkpoint not found. Re-run the setup cell or download from:\n'
        'https://drive.google.com/drive/folders/1eyxwTykveOsbY3XybfIVmWsd1kFF4AAJ'
    )

with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

print('Loading TangoFlux model...')
model = TangoFlux(config=cfg['model'])
model.load_state_dict(load_file(CKPT_PATH), strict=False)
model.to(device).eval()
for p in model.text_encoder.parameters():
    p.requires_grad = False
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model loaded ({trainable/1e6:.0f}M trainable params)')

print('Loading Stable Audio VAE...')
hf_token = None
if IS_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
vae = AutoencoderOobleck.from_pretrained(
    'stabilityai/stable-audio-open-1.0', subfolder='vae', token=hf_token
).to(device).eval()
print('VAE loaded.')

In [ ]:
SR = 44100
CHUNK_DUR = 30
CHUNK_SAMPLES = SR * CHUNK_DUR

def load_audio(path, sr=SR):
    """Load audio, force stereo, resample to target sr."""
    wav, orig_sr = torchaudio.load(path)
    if orig_sr != sr:
        wav = torchaudio.functional.resample(wav, orig_sr, sr)
    if wav.shape[0] == 1:
        wav = wav.repeat(2, 1)
    elif wav.shape[0] > 2:
        wav = wav[:2]
    return wav

def degrade_audio(wav, codec='mp3', bitrate=64000):
    """Apply codec degradation via ffmpeg temp file."""
    import tempfile, subprocess
    with tempfile.NamedTemporaryFile(suffix='.wav') as tmp_in, \
         tempfile.NamedTemporaryFile(suffix=f'.{codec}') as tmp_enc, \
         tempfile.NamedTemporaryFile(suffix='.wav') as tmp_out:
        torchaudio.save(tmp_in.name, wav, SR)
        subprocess.run(['ffmpeg', '-y', '-i', tmp_in.name, '-b:a', str(bitrate), tmp_enc.name], capture_output=True, check=True)
        subprocess.run(['ffmpeg', '-y', '-i', tmp_enc.name, tmp_out.name], capture_output=True, check=True)
        degraded, _ = torchaudio.load(tmp_out.name)
    if degraded.shape[0] == 1:
        degraded = degraded.repeat(2, 1)
    return degraded

@torch.no_grad()
def restore_audio(wav_degraded, prompt, num_steps=10, guidance_scale=1.0):
    """VAE encode -> flow-matching inference -> VAE decode."""
    T = wav_degraded.shape[1]
    if T < CHUNK_SAMPLES:
        wav_degraded = torch.nn.functional.pad(wav_degraded, (0, CHUNK_SAMPLES - T))
    elif T > CHUNK_SAMPLES:
        wav_degraded = wav_degraded[:, :CHUNK_SAMPLES]
    z = vae.encode(wav_degraded.unsqueeze(0).to(device)).latent_dist.mode()
    z = z.transpose(1, 2)
    result = model.inference_flow(
        z, prompt, audiocond_latents=None,
        num_inference_steps=num_steps, guidance_scale=guidance_scale,
        duration=CHUNK_DUR, seed=42, disable_progress=True,
    )
    restored = vae.decode(result.transpose(1, 2)).sample
    return torch.clamp(restored.squeeze(0).cpu(), -1.0, 1.0)[:, :T]

def plot_spectrograms(original, degraded, restored, sr=SR, title=''):
    """Plot three mel spectrograms side by side."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for ax, wav, label in zip(axes, [original, degraded, restored], ['Original', 'Degraded', 'Restored']):
        mono = wav[0].numpy() if wav.dim() == 2 else wav.numpy()
        S = librosa.feature.melspectrogram(y=mono, sr=sr, n_mels=128, fmax=sr//2)
        librosa.display.specshow(librosa.power_to_db(S, ref=np.max), sr=sr, x_axis='time', y_axis='mel', ax=ax, fmax=sr//2)
        ax.set_title(label, fontsize=13)
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

def compute_sdr(reference, estimate):
    ref = reference[0].numpy() if reference.dim() == 2 else reference.numpy()
    est = estimate[0].numpy() if estimate.dim() == 2 else estimate.numpy()
    n = min(len(ref), len(est)); ref, est = ref[:n], est[:n]
    noise = est - ref
    return 10 * np.log10(np.sum(ref**2) / (np.sum(noise**2) + 1e-10))

def compute_sisnr(reference, estimate):
    ref = reference[0].numpy() if reference.dim() == 2 else reference.numpy()
    est = estimate[0].numpy() if estimate.dim() == 2 else estimate.numpy()
    n = min(len(ref), len(est)); ref, est = ref[:n], est[:n]
    ref, est = ref - np.mean(ref), est - np.mean(est)
    s_target = np.dot(est, ref) * ref / (np.dot(ref, ref) + 1e-10)
    e_noise = est - s_target
    return 10 * np.log10(np.sum(s_target**2) / (np.sum(e_noise**2) + 1e-10))

print('Helper functions defined.')

In [ ]:
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
CLEAN_DIR = os.path.join(DATA_DIR, 'clean')

candidates = []
for search_dir in [CLEAN_DIR, DATA_DIR, os.path.join(DATA_DIR, 'clean_clips')]:
    if os.path.isdir(search_dir):
        candidates += glob.glob(os.path.join(search_dir, '*.wav'))
        candidates += glob.glob(os.path.join(search_dir, '*.flac'))

if IS_COLAB and not candidates:
    print('No audio found. Upload a WAV file:')
    from google.colab import files
    uploaded = files.upload()
    TEST_AUDIO = list(uploaded.keys())[0]
elif candidates:
    TEST_AUDIO = candidates[0]
else:
    raise FileNotFoundError(f'No audio found. Place a .wav or .flac in {CLEAN_DIR}')

original = load_audio(TEST_AUDIO)
if original.shape[1] > CHUNK_SAMPLES:
    original = original[:, :CHUNK_SAMPLES]
print(f'Test audio: {os.path.basename(TEST_AUDIO)}')
print(f'Shape: {original.shape}, Duration: {original.shape[1]/SR:.1f}s')

In [ ]:
bitrates = [32000, 64000, 128000]
results = {}

for br in bitrates:
    br_k = br // 1000
    print(f'\n{"=" * 50}  {br_k} kbps MP3  {"=" * 50}')

    degraded = degrade_audio(original, 'mp3', br)
    n = min(original.shape[1], degraded.shape[1])
    degraded, orig_trim = degraded[:, :n], original[:, :n]

    prompt = f'restore audio compressed at {br_k}kbps MP3'
    print(f'Prompt: \"{prompt}\"')
    restored = restore_audio(degraded, prompt)[:, :n]

    sdr_deg  = compute_sdr(orig_trim, degraded)
    sdr_res  = compute_sdr(orig_trim, restored)
    sisnr_deg = compute_sisnr(orig_trim, degraded)
    sisnr_res = compute_sisnr(orig_trim, restored)
    results[br_k] = dict(sdr_deg=sdr_deg, sdr_res=sdr_res, sisnr_deg=sisnr_deg, sisnr_res=sisnr_res)

    print(f'SDR:    {sdr_deg:.2f} -> {sdr_res:.2f} dB  ({sdr_res-sdr_deg:+.2f})')
    print(f'SI-SNR: {sisnr_deg:.2f} -> {sisnr_res:.2f} dB  ({sisnr_res-sisnr_deg:+.2f})')
    plot_spectrograms(orig_trim, degraded, restored, title=f'MP3 @ {br_k} kbps')
    display(Audio(orig_trim[0].numpy(), rate=SR))
    display(Audio(degraded[0].numpy(), rate=SR))
    display(Audio(restored[0].numpy(), rate=SR))

# Summary table
print(f'\n{"="*70}')
print(f'{"Bitrate":>10} | {"SDR Degraded":>14} | {"SDR Restored":>14} | {"SDR Gain":>10}')
print(f'{"":>10} | {"SI-SNR Deg":>14} | {"SI-SNR Res":>14} | {"SI-SNR Gain":>10}')
print('-'*70)
for br_k, m in results.items():
    print(f'{br_k:>8} k | {m["sdr_deg"]:>12.2f} dB | {m["sdr_res"]:>12.2f} dB | {m["sdr_res"]-m["sdr_deg"]:>+8.2f} dB')
    print(f'{"":>10} | {m["sisnr_deg"]:>12.2f} dB | {m["sisnr_res"]:>12.2f} dB | {m["sisnr_res"]-m["sisnr_deg"]:>+8.2f} dB')
    print('-'*70)
print('='*70)

In [ ]:
degraded_64k = degrade_audio(original[:, :CHUNK_SAMPLES], 'mp3', 64000)

prompts_to_test = [
    'remove MP3 compression artifacts',
    'restore audio compressed at 64kbps MP3',
    'enhance low bitrate audio to high fidelity',
    'make this audio sound better',
    '',
]

print('Degraded audio: 64 kbps MP3')
print('Testing different text prompts:\n')

for prompt in prompts_to_test:
    label = prompt if prompt else '(empty / unconditional)'
    restored = restore_audio(degraded_64k, prompt)
    n = min(original.shape[1], restored.shape[1])
    sdr = compute_sdr(original[:, :n], restored[:, :n])
    print(f'Prompt: \"{label}\"  |  SDR: {sdr:.2f} dB')
    display(Audio(restored[0, :n].numpy(), rate=SR))